# Numerical Computation of The Bayes Update Rule for Bayesian Adaptive Filtering

This notebook aims to implement through numerical integration the unidimensional bayesian update rule for general distributions in both the prior and the likelihood.

$$ f_m(\theta_{t,m} | y_{1:t}) \propto f_m(\theta_{t,m} | y_{1:t-1})\,f_{\zeta_m}\!\big(y_t - x_{t,m}\theta_{t,m}\big) $$

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
from numba import njit
from numba_progress import ProgressBar

from filters import (
    # parameter dtypes
    NLMS_params, sKF_params, sKF_L_params, skf_int_params, skf_L_int_params,
    # signal / environment helpers
    autocorr_matrix_calc, autocorr_matrix_estimate, AR_settling_time, std_behavior,
    # algorithms
    NLMS_algorithm, sKF_algorithm, sKF_L_algorithm, sKF_L_exact_algorithm,
    sKF_integral_algorithm, sKF_L_integral_algorithm,
    # monte carlo driver
    MC_Simulations_Modular_Variance,
)

## Framework

All filter implementations, parameter dtypes and Monte Carlo drivers live in [`filters.py`](filters.py).

## Simulations

### Monte Carlo Simulations

#### Bayesian Numerical Filter

##### Gaussian Likelihood Simulations

In [ ]:
L = 3
ho = np.sinc(np.linspace(0,1.5,L))
ho = ho/np.linalg.norm(ho) # ground truth
h0 = np.zeros(L)
var_x = 1
var_v = 1e-3
#AR = np.array([1.0, 0.0])
AR = np.array([1.0, -0.6, 0.85])
#AR = np.array([1.0, -0.9, 0.95, -0.8, 0.8])

Rxx_est = Rxx = autocorr_matrix_calc(AR, 1, M = L)
eig = np.linalg.eigvals(Rxx)
chi = np.max(eig)/np.min(eig)
print(f'Eigenvalue spread: {chi:.4} \n')

##### Bayesian Likelihood Simulations


In [ ]:
NR = 1
N = int(200)

epsilon = 0.01
var_theta_0 = 2 # v_tilde_0
var_eta = 10*var_v
dx_factor = 1/10
min_std_deviations = 5

Algorithms = [sKF_integral_algorithm, sKF_algorithm]

skf_parameters = np.void((f"sKF_integral",
                          epsilon,
                          var_theta_0,
                          var_eta,
                          dx_factor,
                          min_std_deviations), dtype=skf_int_params)

sKF_parameters_paper = np.void(("sKF_paper", epsilon, var_eta, var_theta_0), dtype=sKF_params)

Alg_Parameters = [skf_parameters, sKF_parameters_paper]

with ProgressBar(total=NR) as PBar:
  MC_measures = MC_Simulations_Modular_Variance(N, NR, ho, var_x, var_v, h0, Algorithms, Alg_Parameters, AR, PBar)



###### Graphic Results

In [ ]:
plt.plot(10*np.log10(MC_measures["sKF_integral"]['Jex']))
plt.plot(10*np.log10(MC_measures["sKF_paper"]['Jex']))
plt.ylabel("EMSE (dB)")
plt.xlabel("Iterations")
plt.title("sKF integral Learning Curve")
plt.show()

In [ ]:
h_num   = MC_measures["sKF_integral"]['h']
h_paper = MC_measures["sKF_paper"]['h']
L = h_num.shape[1]
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
%config InlineBackend.figure_format = 'svg'

plt.figure(figsize=(15, 10))
for k in range(L):
  c = colors[k % len(colors)]
  plt.plot(h_num[:, k],   color=c, linestyle='-',  label=f"$w_{k}$ numerical")
  plt.plot(h_paper[:, k], color=c, linestyle='--', label=f"$w_{k}$ paper")
  plt.axhline(ho[k], color=c, linestyle=':', linewidth=1)

plt.plot([], [], color='k', linestyle=':', linewidth=1, label="true $h_k$")
plt.ylabel("w")
plt.xlabel("Iterations")
plt.title("sKF weights: numerical integration vs paper recursion")
plt.legend(ncol=L, fontsize=9)
plt.grid(alpha=0.3)
# plt.savefig("figures/skf_weights.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
h_num   = MC_measures["sKF_integral"]['h']
#h_L_num  = MC_measures["sKF_L_paper"]['h']
#h_L_50_51   = MC_measures["sKF_L_paper"]['h']
h_paper = MC_measures["sKF_paper"]['h']
L = h_num.shape[1]
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
%config InlineBackend.figure_format = 'svg'

plt.figure(figsize=(15, 10))
for k in range(L):
  c = colors[k % len(colors)]
  plt.plot(h_num[:, k],   color=c, linestyle='-',  label=f"$w_{k}$ numerical")
  plt.plot(h_paper[:, k], color=c, linestyle='--', label=f"$w_{k}$ paper")
  #plt.plot(h_L_50_51[:, k], color='k', linestyle='-.', label=f"$w_{k}$ paper Laplace")
  plt.axhline(ho[k], color=c, linestyle=':', linewidth=1)

plt.plot([], [], color='k', linestyle=':', linewidth=1, label="true $h_k$")
plt.ylabel("w")
plt.xlabel("Iterations")
plt.title("sKF weights: numerical integration vs paper recursion")
plt.legend(ncol=L, fontsize=9)
plt.grid(alpha=0.3)
# plt.savefig("figures/skf_weights.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
h_num   = MC_measures["sKF_integral"]['h']
h_paper = MC_measures["sKF_paper"]['h']
L = h_num.shape[1]
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
%config InlineBackend.figure_format = 'svg'

plt.figure(figsize=(15, 10))
for k in range(L):
  c = colors[k % len(colors)]
  plt.plot(10*np.log10(np.abs(h_num[:, k] - h_paper[:,k])/np.abs(h_paper[:,k])),
           color=c, linestyle='-',
           label=f"$w_{k}$ numerical")

plt.plot([], [], color='k', linestyle=':', linewidth=1, label="true $h_k$")
plt.ylabel("weights difference (numerical-paper)/paper [dB]")
plt.xlabel("Iterations")
plt.title("sKF weights: numerical integration vs paper recursion")
plt.legend(ncol=L, fontsize=9)
plt.grid(alpha=0.3)
plt.show()

In [ ]:
v_num   = MC_measures["sKF_integral"]['var']
v_paper = MC_measures["sKF_paper"]['var']
L = v_num.shape[1]
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
%config InlineBackend.figure_format = 'svg'

plt.figure(figsize=(9, 5))

plt.plot(v_num[:, 0],   color=c, linestyle='-',  label=f"v numerical")
plt.plot(v_paper[:, 0], color=c, linestyle='--', label=f"v paper")

#plt.plot([], [], color='k', linestyle=':', linewidth=1, label="true $h_k$")
plt.ylabel("v")
plt.xlabel("Iterations")
plt.title("sKF variance: numerical integration vs paper recursion")
plt.legend(ncol=L, fontsize=9)
plt.grid(alpha=0.3)
plt.show()

In [ ]:
v_num   = MC_measures["sKF_integral"]['var']
v_paper = MC_measures["sKF_paper"]['var']
L = v_num.shape[1]
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
%config InlineBackend.figure_format = 'svg'

plt.figure(figsize=(9, 5))

plt.plot(10*np.log10(np.abs(v_num[:, 0] - v_paper[:, 0])/np.abs(v_paper[:, 0])), color=c, linestyle='--', label=f"v paper")

plt.ylabel("(v_num-v_paper)/v_paper [dB]")
plt.xlabel("Iterations")
plt.title("sKF variance difference: numerical integration vs paper recursion LOG")
plt.legend(ncol=L, fontsize=9)
plt.grid(alpha=0.3)
plt.show()

##### Laplacian Likelihood

In [ ]:
L = 3
ho = np.sinc(np.linspace(0, 1.5, L))
ho = ho / np.linalg.norm(ho)
h0 = np.zeros(L)
var_x = 1
var_v = 1e-3
# AR = np.array([1.0, 0.0])
AR = np.array([1.0, -0.6, 0.85])
# AR = np.array([1.0, -0.9, 0.95, -0.8, 0.8])

Rxx = autocorr_matrix_calc(AR, 1, M = L)
eig = np.linalg.eigvals(Rxx)
chi = np.max(eig) / np.min(eig)
print(f"Eigenvalue spread: {chi:.4} \n")

In [ ]:
NR = 1
N = int(200)

epsilon = 0.01
var_theta_0 = 2  # v_tilde_0
b_eta = 5 * np.sqrt(var_v)
dx_factor = 1/10
min_std_deviations = 5

Algorithms = [sKF_L_integral_algorithm, sKF_L_algorithm, sKF_L_exact_algorithm]

skf_L_parameters = np.void(
    ("sKF_L_integral", epsilon, var_theta_0, b_eta, dx_factor, min_std_deviations),
    dtype=skf_L_int_params,
)

sKF_L_parameters_minorized = np.void(
    ("sKF_L_minorized", epsilon, b_eta, var_theta_0), dtype=sKF_L_params
)

sKF_L_parameters_exact = np.void(
    ("sKF_L_exact", epsilon, b_eta, var_theta_0), dtype=sKF_L_params
)

Alg_Parameters = [skf_L_parameters, sKF_L_parameters_minorized, sKF_L_parameters_exact]

with ProgressBar(total=NR) as PBar:
    MC_measures = MC_Simulations_Modular_Variance(
        N, NR, ho, var_x, var_v, h0, Algorithms, Alg_Parameters, AR, PBar
    )

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(10 * np.log10(MC_measures["sKF_L_minorized"]["Jex"]))
plt.plot(10 * np.log10(MC_measures["sKF_L_exact"]["Jex"]))
plt.plot(10 * np.log10(MC_measures["sKF_L_integral"]["Jex"]))

plt.ylabel("EMSE (dB)")
plt.xlabel("Iterations")
plt.title("sKF integral Learning Curve")
plt.show()

In [ ]:
h_L_num   = MC_measures["sKF_L_integral"]['h']
h_L_minorized = MC_measures["sKF_L_minorized"]['h']
h_L_exact = MC_measures["sKF_L_exact"]['h']

L = h_L_num.shape[1]
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
%config InlineBackend.figure_format = 'svg'

plt.figure(figsize=(15, 10))
for k in range(L):
  c = colors[k % len(colors)]
  plt.plot(h_L_num[:, k],   color=c, linestyle='-',  label=f"$w_{k}$ numerical")
  plt.plot(h_L_minorized[:, k], color=c, linestyle='--', linewidth=1, label=f"$w_{k}$ minorized")
  plt.plot(h_L_exact[:, k], color=c, linestyle='-.', label=f"$w_{k}$ exact")
  plt.axhline(ho[k], color=c, linestyle=':', linewidth=1)

plt.plot([], [], color='k', linestyle=':', linewidth=1, label="true $h_k$")
plt.ylabel("w")
plt.xlabel("Iterations")
plt.title("sKF-L weights: numerical integration vs paper recursion")

# --- PLOTTING THE PARAMETERS IN THE LEGEND ---
ax = plt.gca()
leg1 = ax.legend(ncol=L, fontsize=9, loc="upper right")
ax.add_artist(leg1)
def fmt(p):
    return "\n".join(f"{k} = {p[k]}" for k in p.dtype.names)
ax.legend(handles=[],
          title=f"integral:\n{fmt(skf_L_parameters)}\n\nminorized and exact:\n{fmt(sKF_L_parameters_minorized)}",
          loc="lower right", fontsize=8, title_fontsize=8)
# --- ---
plt.grid(alpha=0.3)
plt.savefig("skf_weights.png", dpi=300, bbox_inches="tight")
plt.show()


##### GRID SEARCH

In [ ]:
import itertools, pickle, os

GRID = {
    "NR":                 [1, 3],
    "epsilon":            [1e-3, 1e-2],
    "var_theta_0":        [0.5, 2.0],
    "var_eta":            [1e-3, 1e-2],
    "dx_factor":          [1.0, 0.2],
    "min_std_deviations": [5, 10],
}

CACHE = "grid_results.pkl"
results = pickle.load(open(CACHE, "rb")) if os.path.exists(CACHE) else {}

@njit
def numba_seed(s): np.random.seed(s)

N=200

for combo in itertools.product(*GRID.values()):
    if combo in results:
        continue
    NR_, eps, vt0, veta, dxf, msd = combo

    p_int   = np.void(("sKF_integral", eps, vt0, veta, dxf, msd), dtype=skf_int_params)
    p_paper = np.void(("sKF_paper",    eps, veta, vt0),           dtype=sKF_params)

    numba_seed(0)
    results[combo] = MC_Simulations_Modular_Variance(
        N, NR_, ho, var_x, var_v, h0,
        [sKF_integral_algorithm, sKF_algorithm],
        [p_int, p_paper], AR)

    pickle.dump(results, open(CACHE, "wb"))   # save each cell, crash-safe
    print(combo, "done")

In [ ]:
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

def grid_plot(draw, title, sharey=True, ncol=2):
    combos = list(results.keys())
    nrow = int(np.ceil(len(combos)/ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(7*ncol, 4*nrow),
                             sharey=sharey, squeeze=False,
                             constrained_layout=True)
    for ax, combo in zip(axes.ravel(), combos):
        draw(ax, results[combo])
        items = [f"{k}={v:g}" for k, v in zip(GRID, combo)]
        ax.set_title(", ".join(items[:3]) + "\n" + ", ".join(items[3:]), fontsize=9)
        ax.grid(alpha=0.3)
    for ax in axes.ravel()[len(combos):]:
        ax.axis("off")
    axes[0, 0].legend(fontsize=7)
    fig.suptitle(title)
    plt.show()


def draw_weights(ax, r):                                    # cell 32
    num, pap = r["sKF_integral"]["h"], r["sKF_paper"]["h"]
    for k in range(num.shape[1]):
        c = colors[k % len(colors)]
        ax.plot(num[:, k], color=c, ls='-',  label=f"$w_{k}$ num")
        ax.plot(pap[:, k], color=c, ls='--', label=f"$w_{k}$ paper")
        ax.axhline(ho[k], color=c, ls=':', lw=1)

def draw_weights_diff(ax, r):                               # cell 33
    num, pap = r["sKF_integral"]["h"], r["sKF_paper"]["h"]
    for k in range(num.shape[1]):
        ax.plot(10*np.log10(np.abs(num[:, k] - pap[:, k]) / np.abs(pap[:, k])),
                color=colors[k % len(colors)], label=f"$w_{k}$")
    ax.axhline(10*np.log10(0.012), ls=":", c="k")           # regressor floor, 1.2%

def draw_var(ax, r):                                        # cell 34
    ax.plot(r["sKF_integral"]["var"][:, 0], '-',  label="v num")
    ax.plot(r["sKF_paper"]["var"][:, 0],    '--', label="v paper")

def draw_var_diff(ax, r):                                   # cell 36
    vn, vp = r["sKF_integral"]["var"][:, 0], r["sKF_paper"]["var"][:, 0]
    ax.plot(10*np.log10(np.abs(vn - vp) / np.abs(vp)))
    ax.axhline(10*np.log10(0.012), ls=":", c="k")


grid_plot(draw_weights,      "sKF weights: numerical integration vs paper recursion")
grid_plot(draw_weights_diff, "sKF weights difference (num-paper)/paper [dB]")
grid_plot(draw_var,          "sKF variance: numerical integration vs paper recursion",
          sharey=False)
grid_plot(draw_var_diff,     "sKF variance difference (num-paper)/paper [dB]")

In [ ]:
DRAWS = [(draw_weights,      "weights: num vs paper"),
         (draw_weights_diff, "weights diff [dB]"),
         (draw_var,          "variance: num vs paper"),
         (draw_var_diff,     "variance diff [dB]")]

def grid_svg(fname="grid.svg", combos=None):
    combos = list(results.keys()) if combos is None else combos
    nrow, ncol = len(combos), len(DRAWS)
    fig, axes = plt.subplots(nrow, ncol, figsize=(7*ncol, 4*nrow),
                             sharey='col', squeeze=False,
                             constrained_layout=True)
    for i, combo in enumerate(combos):
        items = [f"{k}={v:g}" for k, v in zip(GRID, combo)]
        label = ", ".join(items[:3]) + "\n" + ", ".join(items[3:])
        for j, (draw, name) in enumerate(DRAWS):
            draw(axes[i, j], results[combo])
            axes[i, j].set_title(f"{name}\n{label}", fontsize=8)
            axes[i, j].grid(alpha=0.3)
    axes[0, 0].legend(fontsize=7)
    fig.savefig(fname, format="svg")
    plt.close(fig)

grid_svg("grid.svg")

In [ ]:
combos = list(results.keys())
ncol = 4; nrow = int(np.ceil(len(combos)/ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(4*ncol, 3*nrow), sharey=True)

for ax, combo in zip(axes.ravel(), combos):
    r = results[combo]
    num, pap = r["sKF_integral"]["h"], r["sKF_paper"]["h"]
    rel = np.linalg.norm(num - pap, axis=1) / np.linalg.norm(pap, axis=1)
    ax.plot(10*np.log10(rel))
    ax.axhline(10*np.log10(0.012), ls=":", c="k")   # regressor floor, 1.2%
    ax.set_title(", ".join(f"{k}={v:g}" for k, v in zip(GRID, combo)), fontsize=7)

In [ ]:
MC_measures["sKF_integral"]['h']